# NegotiateEnv Training - Direct (No Server)

**Team**: Kushal Adhyaru & Mayuka Kothuru  
**Repository**: https://github.com/kushal511/saas-negotiation-env

This notebook uses the environment directly without HTTP server - simpler and more reliable!

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/kushal511/saas-negotiation-env.git"
REPO_NAME = "saas-negotiation-env"

if os.path.exists(REPO_NAME):
    print("Repository exists. Pulling latest...")
    %cd {REPO_NAME}
    !git fetch origin
    !git reset --hard origin/main
    !git pull
else:
    print("Cloning repository...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}

print("Repository ready!")

## 3. Install Dependencies

In [ ]:
# Install environment package
!pip install -q -e .

# Install training dependencies
!pip uninstall -y -q vllm 2>/dev/null || true
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.11.0" transformers accelerate peft datasets

print("All dependencies installed!")

## 4. Baseline Performance Evaluation

In [ ]:
from negotiate_env.server.environment import NegotiateEnvironment
from negotiate_env.models import NegotiateAction

def run_baseline_episodes(num_episodes=100, difficulty="hard"):
    """Run baseline episodes with rule-based policy."""
    env = NegotiateEnvironment(difficulty=difficulty, use_hf_dataset=True)
    rewards = []
    successes = 0
    
    for ep in range(num_episodes):
        obs = env.reset()
        done = False
        turn = 0
        
        while not done and turn < 50:
            turn += 1
            
            current_price = obs.current_offer.get("price_per_seat", obs.your_max_price)
            current_length = obs.current_offer.get("contract_length", 2.0)
            current_cap = obs.current_offer.get("annual_increase_cap", 7.0)
            
            if (current_price <= obs.your_max_price and 
                current_length <= obs.your_max_length and 
                current_cap <= obs.your_max_cap):
                action = NegotiateAction(action_type="accept", message="This works for us.")
            elif turn == 1:
                action = NegotiateAction(action_type="probe", message="What's your best price?")
            else:
                action = NegotiateAction(
                    action_type="counter",
                    price_per_seat=obs.your_max_price * 0.85,
                    contract_length=min(2.0, obs.your_max_length),
                    annual_increase_cap=min(5.0, obs.your_max_cap),
                    message="Can we negotiate?"
                )
            
            obs = env.step(action)
            done = obs.done
        
        rewards.append(obs.reward)
        if obs.reward > 0:
            successes += 1
        
        if (ep + 1) % 20 == 0:
            print(f"  Completed {ep + 1}/{num_episodes} episodes...")
    
    return rewards, successes

print("="*60)
print("BASELINE PERFORMANCE (Rule-Based Policy)")
print("="*60)
print("\nRunning 100 episodes with 50-turn planning...")
print()

baseline_rewards, baseline_successes = run_baseline_episodes(100, "hard")

avg_reward = sum(baseline_rewards) / len(baseline_rewards)
success_rate = baseline_successes / len(baseline_rewards) * 100

print("\n" + "="*60)
print("BASELINE RESULTS")
print("="*60)
print(f"Episodes: {len(baseline_rewards)}")
print(f"Average Reward: {avg_reward:.4f}")
print(f"Success Rate: {success_rate:.1f}%")
print(f"Best Reward: {max(baseline_rewards):.4f}")
print(f"Worst Reward: {min(baseline_rewards):.4f}")
print("\nThis is your benchmark to beat with training!")
print("="*60)

## 5. Run Training

In [ ]:
print("Starting training (direct environment, no server needed)...")
print()

!python train_negotiate_direct.py \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-long-horizon-output \
    --num-episodes 1000 \
    --max-turns 50 \
    --difficulty hard

print("\nTraining complete!")
print("Model saved to: negotiate-long-horizon-output/")

## 6. Training Results

In [ ]:
import json
import os

print("="*60)
print("TRAINING RESULTS")
print("="*60)

trainer_state_path = "negotiate-long-horizon-output/trainer_state.json"

if os.path.exists(trainer_state_path):
    with open(trainer_state_path) as f:
        state = json.load(f)
    
    log_history = state.get("log_history", [])
    rewards = []
    for entry in log_history:
        reward = entry.get("env_reward") or entry.get("reward") or entry.get("train/env_reward") or entry.get("train/reward")
        if reward is not None:
            rewards.append(float(reward))
    
    if rewards:
        print(f"\nEpisodes: {len(rewards)}")
        print(f"Initial Reward: {rewards[0]:.4f}")
        print(f"Final Reward: {rewards[-1]:.4f}")
        print(f"Best Reward: {max(rewards):.4f}")
        print(f"Average Reward: {sum(rewards)/len(rewards):.4f}")
        
        improvement = rewards[-1] - rewards[0]
        improvement_pct = (improvement / abs(rewards[0]) * 100) if rewards[0] != 0 else 0
        print(f"\nImprovement: {improvement:+.4f} ({improvement_pct:+.1f}%)")
        
        # Compare to baseline
        try:
            baseline_avg = avg_reward
            vs_baseline = rewards[-1] - baseline_avg
            vs_baseline_pct = (vs_baseline / abs(baseline_avg) * 100) if baseline_avg != 0 else 0
            print(f"\nBaseline: {baseline_avg:.4f}")
            print(f"Trained: {rewards[-1]:.4f}")
            print(f"vs Baseline: {vs_baseline:+.4f} ({vs_baseline_pct:+.1f}%)")
        except:
            pass
        
        print(f"\nTraining Milestones:")
        milestones = [0, len(rewards)//4, len(rewards)//2, 3*len(rewards)//4, len(rewards)-1]
        for idx in milestones:
            if idx < len(rewards):
                print(f"Episode {idx:4d}: {rewards[idx]:.4f}")
    else:
        print("\nNo reward data found")
else:
    print("\nTraining state not found")

print("\n" + "="*60)

## 7. Save Model to HuggingFace

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('Huggingface_Token')
    login(token=hf_token)
    print("Logged in to HuggingFace!")
except:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
output_dir = "negotiate-long-horizon-output"
repo_id = "KushalAdhyaru/negotiate-env-long-horizon-1000ep"

if os.path.exists(output_dir):
    print(f"Uploading to {repo_id}...")
    try:
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            repo_type="model",
        )
        print(f"\nModel uploaded!")
        print(f"View at: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"Upload failed: {e}")
else:
    print("Output directory not found")

## Summary

**Statement 2: Super Long-Horizon Planning**

Implemented Features:
- Extended episodes (50 turns)
- 300+ scattered instructions
- 8-stage sales workflow
- Multi-deal negotiation
- Sparse rewards
- Proportional turn penalty

**Training Method**: Direct environment (no HTTP server needed)

**Team**: Kushal Adhyaru & Mayuka Kothuru  
**Repository**: https://github.com/kushal511/saas-negotiation-env